# Organized research notebook

Cell source is retained from the supplied research archive. Saved outputs and credentials have been removed. Run the setup cell first. Check model, SAE, file paths, and run settings in the cells before starting a GPU experiment. This notebook has been checked statically, not rerun against the paper results. See `docs/RELEASE_REVIEW.md` for unresolved items.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


# Emotion-vector steering: external-dataset generalizability

This notebook preserves the supplied notebook's concept-vector extraction and scaler-training process on the **main training dataset**. It then evaluates the unsteered and steered model on two external English emotion datasets:

- `data/emoevent_en.csv` (tab-separated; `tweet` is normalized to `input_text`; the out-of-scope `others` label is filtered)
- `data/xed-en.csv` (comma-separated; `text` is normalized to `input_text`)

To make the unsteered baseline directly comparable to `feature_steering_utils.py`, both external datasets use its exact zero-shot prompt: `In this text: {text}, the emotion implied is:` and classification tokenization is truncated to 512 tokens.

The evaluation reports accuracy, macro F1, and F1 for each of the six emotions. It also saves row-level predictions and a long-form F1 table. Change only `MODEL_NAME` in the configuration cell; `LAYER_ID` is selected from the required model-layer mapping automatically.

Use a fresh kernel before the first run so the Hugging Face cache variables are set before NNsight and Transformers are imported.


In [ ]:
import os

def setup_hf_cache(cache_root: str) -> None:
    os.environ["HF_HOME"] = cache_root
    os.environ["HF_DATASETS_CACHE"] = f"{cache_root}/datasets"
    os.environ["TRANSFORMERS_CACHE"] = f"{cache_root}/transformers"
    os.environ["HUGGINGFACE_HUB_CACHE"] = f"{cache_root}/hub"
    os.environ["TORCH_HOME"] = f"{cache_root}/torch"
    os.environ["XDG_CACHE_HOME"] = cache_root

HF_CACHE = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
# This must run before importing NNsight/Transformers.
setup_hf_cache(HF_CACHE)

print("Configuration loaded.")

In [ ]:
import argparse
import gc
import hashlib
import json
import math
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from nnsight import LanguageModel
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

print("Imports finished.")


In [ ]:
# ============================================================
# Notebook configuration
# Choose one supported MODEL_NAME, then run all cells.
# LAYER_ID is selected automatically from MODEL_LAYERS.
# ============================================================

MODEL_LAYERS = {
    "google/gemma-2-2b": 22,
    "google/gemma-2-9b": 39,
    "google/gemma-3-12b-it": 41,
    "google/gemma-3-12b-pt": 45,
    "google/gemma-3-4b-it": 25,
    "google/gemma-3-4b-pt": 29,
    "meta-llama/Llama-3.1-8B": 30,
}

MODEL_NAME = "meta-llama/Llama-3.1-8B"

if MODEL_NAME not in MODEL_LAYERS:
    raise ValueError(
        f"Unsupported MODEL_NAME={MODEL_NAME!r}. "
        f"Choose one of: {list(MODEL_LAYERS)}"
    )

LAYER_ID = MODEL_LAYERS[MODEL_NAME]

# Main dataset: used for hidden-state extraction, concept-vector construction,
# and emotion-scaler training exactly as in the supplied notebook.
TRAIN_CSV = "data/train_df.csv"

# External datasets: evaluation only. They never affect vector extraction or
# scaler training.
EXTERNAL_DATASETS = {
    "emoevent": {
        "path": "data/emoevent_en.csv",
        "sep": "\t",
        "text_col": "tweet",
        "label_col": "emotion",
        "apply_prompt_template": True,
    },
    "xed": {
        "path": "data/xed-en.csv",
        "sep": ",",
        "text_col": "text",
        "label_col": "emotion",
        "apply_prompt_template": True,
    },
}

OUTPUT_DIR = "data"
TEXT_COL = "input_text"
LABEL_COL = "emotion"

HIDDEN_BATCH_SIZE = 8
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
CLASSIFICATION_MAX_LENGTH = 512

NUM_STEPS = 600
LEARNING_RATE = 1e-2
SCALE_REGULARIZATION = 1e-3
MIN_SCALE = 0.05
MAX_SCALE = 20.0

SEED = 42
LOG_EVERY = 10
OVERWRITE_HIDDEN = False

# Optional general-language perplexity diagnostic retained from the supplied
# comparable-perplexity notebook. It is off by default because this notebook's
# primary outcome is external-dataset F1.
RUN_PERPLEXITY_CHECK = False
USE_WIKITEXT = True
WIKITEXT_DATASET_NAME = "wikitext"
WIKITEXT_DATASET_CONFIG = "wikitext-103-raw-v1"
WIKITEXT_SPLIT = "test"
WIKITEXT_CACHE_DIR = os.environ.get("HF_DATASETS_CACHE", os.path.expanduser("~/.cache/huggingface/datasets"))
PPL_MIN_CHARS = 100
PPL_MAX_TEXTS = 500
PPL_BATCH_SIZE = 8
PPL_MAX_LENGTH = 512

print(f"Model: {MODEL_NAME}")
print(f"Layer: {LAYER_ID}")
print(f"External datasets: {list(EXTERNAL_DATASETS)}")


In [ ]:
# ============================================================
# Reproducibility and cache
# ============================================================


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup_memory() -> None:
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def model_safe_name(model_name: str) -> str:
    return model_name.replace("/", "-")


def get_device_name() -> str:
    if torch.cuda.is_available():
        return "cuda"

    if torch.backends.mps.is_available():
        return "mps"

    return "cpu"


def choose_dtype(device_name: str) -> torch.dtype:
    if device_name == "cuda":
        return torch.bfloat16

    return torch.float32


# ============================================================
# Data validation
# ============================================================

def load_and_validate_dataframe(
    csv_path: str,
    text_col: str,
    label_col: str,
    emotions: Sequence[str],
    require_all_emotions: bool,
) -> pd.DataFrame:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    df = pd.read_csv(csv_path).reset_index(drop=True)

    missing_columns = [
        column
        for column in [text_col, label_col]
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{csv_path} is missing columns {missing_columns}. "
            f"Available columns: {df.columns.tolist()}"
        )

    if df[text_col].isna().any():
        raise ValueError(f"{csv_path}: '{text_col}' contains missing values.")

    if df[label_col].isna().any():
        raise ValueError(f"{csv_path}: '{label_col}' contains missing values.")

    df[text_col] = df[text_col].astype(str)
    df[label_col] = df[label_col].astype(str).str.strip().str.lower()

    unknown_labels = sorted(set(df[label_col]) - set(emotions))

    if unknown_labels:
        raise ValueError(
            f"{csv_path} contains unknown emotion labels: {unknown_labels}"
        )

    if require_all_emotions:
        missing_emotions = [
            emotion
            for emotion in emotions
            if emotion not in set(df[label_col])
        ]

        if missing_emotions:
            raise ValueError(
                f"{csv_path} has no training examples for: {missing_emotions}"
            )

    return df


def dataframe_fingerprint(
    df: pd.DataFrame,
    columns: Sequence[str],
) -> str:
    """Create a stable content hash for cache validation."""
    hasher = hashlib.sha256()

    for row in df[list(columns)].itertuples(index=False, name=None):
        for value in row:
            hasher.update(str(value).encode("utf-8"))
            hasher.update(b"\x1f")

        hasher.update(b"\x1e")

    return hasher.hexdigest()


# ============================================================
# Model utilities
# ============================================================

def load_llm(
    model_name: str,
    device_map: str,
    dtype: torch.dtype,
):
    llm = LanguageModel(
        model_name,
        device_map="cuda:0",
        dtype=dtype,
        dispatch=True,
    )

    llm.tokenizer.padding_side = "left"

    if llm.tokenizer.pad_token_id is None:
        if llm.tokenizer.eos_token_id is None:
            raise ValueError(
                "Tokenizer has neither a pad token nor an EOS token."
            )

        llm.tokenizer.pad_token = llm.tokenizer.eos_token

    underlying_model = getattr(llm, "_model", None)

    if underlying_model is not None:
        try:
            underlying_model.config.use_cache = False
        except Exception:
            pass

        try:
            underlying_model.eval()
        except Exception:
            pass

    return llm


def get_model_layers_and_dims(llm):
    """Return the real HF layer modules, layer count, and hidden size."""
    model = getattr(llm, "_model", None)
    if model is None:
        raise RuntimeError("Could not access the underlying Hugging Face model.")

    layers = get_underlying_transformer_layers(llm)
    config = model.config

    if hasattr(config, "get_text_config"):
        try:
            config = config.get_text_config()
        except Exception:
            pass
    elif hasattr(config, "text_config"):
        config = config.text_config

    n_layers = len(layers)
    hidden_dim = int(config.hidden_size)
    return layers, n_layers, hidden_dim


def validate_layer_id(layer_id: int, n_layers: int) -> None:
    if layer_id < 0 or layer_id >= n_layers:
        raise ValueError(
            f"layer_id={layer_id} is invalid. "
            f"The model has layers 0 through {n_layers - 1}."
        )


def freeze_llm_parameters(llm) -> None:
    underlying_model = getattr(llm, "_model", None)

    if underlying_model is None:
        raise RuntimeError(
            "Could not access the underlying PyTorch model through llm._model."
        )

    parameter_count = 0

    for parameter in underlying_model.parameters():
        parameter.requires_grad_(False)
        parameter_count += 1

    underlying_model.eval()
    print(f"Frozen {parameter_count} model parameter tensors.")


def get_model_device(llm) -> torch.device:
    # NNsight may expose only meta-device parameters when the model is
    # dispatched, so place steering tensors explicitly on the first GPU.
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required, but cuda:0 is unavailable.")

    return torch.device("cuda:0")


# ============================================================
# Emotion output tokens
# ============================================================

def build_emotion_token_maps(
    llm,
    emotions: Sequence[str],
    verbose: bool = True,
) -> Tuple[Dict[int, str], Dict[str, int]]:
    """
    Match feature_steering_utils.py exactly: encode each space-prefixed
    emotion with tokenizer defaults and score its final token ID.
    """
    token_to_emotion: Dict[int, str] = {}
    emotion_to_token: Dict[str, int] = {}

    for emotion in emotions:
        token_ids = llm.tokenizer.encode(" " + emotion)
        if len(token_ids) == 0:
            raise ValueError(f"Could not tokenize emotion: {emotion}")
        token_id = int(token_ids[-1])

        if token_id in token_to_emotion:
            raise ValueError(
                f"Token ID {token_id} is shared by "
                f"'{token_to_emotion[token_id]}' and '{emotion}'."
            )

        token_to_emotion[token_id] = emotion
        emotion_to_token[emotion] = token_id

        if verbose:
            pieces = [llm.tokenizer.decode([int(x)]) for x in token_ids]
            print(
                f"{emotion:>8}: {token_ids} -> pieces={pieces} -> "
                f"using last token {token_id}"
            )

    return token_to_emotion, emotion_to_token


# ============================================================
# Selected-layer hidden-state extraction
# ============================================================



def get_hidden_tensor_from_layer_output(layer_output):
    """Handle decoder layers that return either a tensor or a tuple."""
    if torch.is_tensor(layer_output):
        return layer_output

    if isinstance(layer_output, (tuple, list)) and layer_output:
        if torch.is_tensor(layer_output[0]):
            return layer_output[0]

    raise TypeError(
        "Unsupported transformer-layer output type: "
        f"{type(layer_output).__name__}"
    )


def replace_hidden_tensor_in_layer_output(layer_output, hidden_states):
    """Return a layer output with its residual tensor replaced."""
    if torch.is_tensor(layer_output):
        return hidden_states

    if isinstance(layer_output, tuple):
        return (hidden_states,) + layer_output[1:]

    if isinstance(layer_output, list):
        return [hidden_states] + layer_output[1:]

    raise TypeError(
        "Unsupported transformer-layer output type: "
        f"{type(layer_output).__name__}"
    )


def tokenize_model_batch(
    llm,
    texts,
    device,
    truncation=False,
    max_length=None,
):
    """Tokenize supplied texts and move inputs to cuda:0."""
    encoded = llm.tokenizer(
        list(texts),
        padding=True,
        truncation=truncation,
        max_length=max_length,
        return_tensors="pt",
    )
    return {
        key: value.to(device)
        for key, value in encoded.items()
        if key in {"input_ids", "attention_mask"}
    }


def extract_selected_layer_hidden_states(
    llm,
    layers,
    df: pd.DataFrame,
    layer_id: int,
    text_col: str,
    batch_size: int,
    hidden_dim: int,
) -> np.ndarray:
    """
    Extract the same final-token residual representation as the source
    notebook, using a native PyTorch hook to avoid NNsight/Transformers
    tracing incompatibilities.
    """
    hidden_states = np.zeros((len(df), hidden_dim), dtype=np.float32)
    model = getattr(llm, "_model", None)
    if model is None:
        raise RuntimeError("Could not access the underlying Hugging Face model.")

    model.eval()
    device = get_model_device(llm)
    captured = {}

    def capture_hook(_module, _inputs, layer_output):
        residual = get_hidden_tensor_from_layer_output(layer_output)
        captured["last_token"] = residual[:, -1, :].detach()

    hook_handle = layers[layer_id].register_forward_hook(capture_hook)

    try:
        for start in tqdm(
            range(0, len(df), batch_size),
            desc=f"Extracting train hidden states at layer {layer_id}",
        ):
            end = min(start + batch_size, len(df))
            batch_inputs = df[text_col].iloc[start:end].astype(str).tolist()
            model_inputs = tokenize_model_batch(llm, batch_inputs, device)

            with torch.no_grad():
                model(**model_inputs, use_cache=False, return_dict=True)

            if "last_token" not in captured:
                raise RuntimeError("The selected-layer hook did not run.")

            batch_hidden = captured.pop("last_token").float().cpu().numpy()
            expected_shape = (end - start, hidden_dim)
            if batch_hidden.shape != expected_shape:
                raise RuntimeError(
                    f"Expected hidden-state shape {expected_shape}, "
                    f"received {batch_hidden.shape}."
                )

            hidden_states[start:end] = batch_hidden
            del model_inputs, batch_hidden
            cleanup_memory()
    finally:
        hook_handle.remove()

    return hidden_states


def load_or_extract_train_hidden_states(
    llm,
    layers,
    train_df: pd.DataFrame,
    layer_id: int,
    text_col: str,
    batch_size: int,
    hidden_dim: int,
    hidden_path: Path,
    metadata_path: Path,
    cache_metadata: Dict[str, object],
    overwrite_hidden: bool,
) -> np.ndarray:
    if (
        hidden_path.exists()
        and metadata_path.exists()
        and not overwrite_hidden
    ):
        with open(metadata_path, "r") as file:
            existing_metadata = json.load(file)

        if existing_metadata != cache_metadata:
            raise ValueError(
                "The cached hidden states were created from different "
                "data or settings. Rerun with --overwrite_hidden."
            )

        hidden_states = np.load(hidden_path)

        expected_shape = (len(train_df), hidden_dim)

        if hidden_states.shape != expected_shape:
            raise ValueError(
                f"Cached hidden states at {hidden_path} have shape "
                f"{hidden_states.shape}, expected {expected_shape}. "
                "Rerun with --overwrite_hidden."
            )

        print(f"Loaded cached hidden states: {hidden_path}")
        return hidden_states.astype(np.float32, copy=False)

    hidden_states = extract_selected_layer_hidden_states(
        llm=llm,
        layers=layers,
        df=train_df,
        layer_id=layer_id,
        text_col=text_col,
        batch_size=batch_size,
        hidden_dim=hidden_dim,
    )

    hidden_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(hidden_path, hidden_states)

    with open(metadata_path, "w") as file:
        json.dump(cache_metadata, file, indent=2)

    print(f"Saved train hidden states: {hidden_path}")

    return hidden_states


# ============================================================
# Contrastive emotion directions
# ============================================================

def compute_contrastive_emotion_vectors(
    train_hidden_states: np.ndarray,
    train_labels: np.ndarray,
    emotions: Sequence[str],
    eps: float = 1e-8,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    For emotion e:
        v_e = normalize(mean(h | y=e) - mean(h | y!=e))

    Returns:
        directions: [num_emotions, hidden_dim]
        center:     [hidden_dim]

    The global training mean is used to center each example before
    projecting it onto the emotion directions.
    """
    if train_hidden_states.ndim != 2:
        raise ValueError(
            "train_hidden_states must have shape [examples, hidden_dim]."
        )

    if len(train_hidden_states) != len(train_labels):
        raise ValueError(
            "The number of hidden states does not match the number of labels."
        )

    center = train_hidden_states.mean(axis=0).astype(np.float32)
    directions = np.zeros(
        (len(emotions), train_hidden_states.shape[1]),
        dtype=np.float32,
    )

    for emotion_index, emotion in enumerate(emotions):
        positive_mask = train_labels == emotion
        negative_mask = train_labels != emotion

        positive_count = int(positive_mask.sum())
        negative_count = int(negative_mask.sum())

        if positive_count == 0 or negative_count == 0:
            raise ValueError(
                f"Cannot construct direction for '{emotion}': "
                f"{positive_count} positive and {negative_count} negative."
            )

        positive_mean = train_hidden_states[positive_mask].mean(axis=0)
        negative_mean = train_hidden_states[negative_mask].mean(axis=0)

        direction = positive_mean - negative_mean
        norm = float(np.linalg.norm(direction))

        if norm <= eps:
            raise ValueError(
                f"The contrastive direction for '{emotion}' has near-zero norm."
            )

        directions[emotion_index] = direction / norm

        print(
            f"Direction {emotion:>8}: "
            f"{positive_count} positive, {negative_count} negative"
        )

    return directions, center


def save_direction_diagnostics(
    directions: np.ndarray,
    emotions: Sequence[str],
    output_dir: Path,
) -> None:
    cosine_similarity = directions @ directions.T

    cosine_df = pd.DataFrame(
        cosine_similarity,
        index=emotions,
        columns=emotions,
    )

    cosine_df.to_csv(
        output_dir / "emotion_direction_cosine_similarity.csv",
        index=True,
    )


# ============================================================
# Steering intervention and evaluation
# ============================================================

def apply_projection_scaling(
    final_hidden_float,
    directions: torch.Tensor,
    center: torch.Tensor,
    scales: torch.Tensor,
):
    """
    Scale the input-specific projection onto every emotion direction:

        z       = h - center
        p_e     = <z, v_e>
        delta   = sum_e (scale_e - 1) p_e v_e
        h'      = h + delta

    scales == 1 produces exactly the unsteered model.
    """
    centered_hidden = final_hidden_float - center.unsqueeze(0)
    projections = centered_hidden @ directions.T
    scaled_change = projections * (scales - 1.0).unsqueeze(0)
    delta = scaled_change @ directions

    return final_hidden_float + delta


def compute_classification_metrics(
    true_labels: Sequence[str],
    predicted_labels: Sequence[str],
    emotions: Sequence[str],
) -> Dict[str, float]:
    metrics: Dict[str, float] = {
        "accuracy": float(
            accuracy_score(true_labels, predicted_labels)
        ),
        "macro_f1": float(
            f1_score(
                true_labels,
                predicted_labels,
                labels=list(emotions),
                average="macro",
                zero_division=0,
            )
        ),
    }

    per_emotion_f1 = f1_score(
        true_labels,
        predicted_labels,
        labels=list(emotions),
        average=None,
        zero_division=0,
    )

    for emotion, score in zip(emotions, per_emotion_f1):
        metrics[f"{emotion}_f1"] = float(score)

    return metrics


def evaluate_emotion_vector_steering(
    llm,
    layers,
    df: pd.DataFrame,
    directions: torch.Tensor,
    center: torch.Tensor,
    scales: torch.Tensor,
    layer_id: int,
    emotions: Sequence[str],
    emotion_token_ids: Sequence[int],
    text_col: str,
    label_col: str,
    batch_size: int,
    pred_col: str,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """Evaluate the identical projection intervention with PyTorch hooks."""
    df_out = df.copy().reset_index(drop=True)
    all_predictions: List[str] = []
    detached_scales = scales.detach()
    model = getattr(llm, "_model", None)
    if model is None:
        raise RuntimeError("Could not access the underlying Hugging Face model.")

    model.eval()
    device = get_model_device(llm)

    def steering_hook(_module, _inputs, layer_output):
        residual = get_hidden_tensor_from_layer_output(layer_output)
        final_hidden = residual[:, -1, :]
        steered_final = apply_projection_scaling(
            final_hidden_float=final_hidden.float(),
            directions=directions,
            center=center,
            scales=detached_scales,
        ).to(final_hidden.dtype)
        modified = torch.cat(
            [residual[:, :-1, :], steered_final.unsqueeze(1)],
            dim=1,
        )
        return replace_hidden_tensor_in_layer_output(layer_output, modified)

    # A scale vector of all ones is the true clean baseline: do not
    # register any intervention hook for the unsteered condition.
    is_unsteered = bool(torch.all(detached_scales == 1.0).item())
    hook_handle = None
    if not is_unsteered:
        hook_handle = layers[layer_id].register_forward_hook(steering_hook)

    try:
        for start in tqdm(
            range(0, len(df_out), batch_size),
            desc=f"Evaluating {pred_col}",
        ):
            end = min(start + batch_size, len(df_out))
            batch_inputs = df_out[text_col].iloc[start:end].astype(str).tolist()
            model_inputs = tokenize_model_batch(
                llm,
                batch_inputs,
                device,
                truncation=True,
                max_length=CLASSIFICATION_MAX_LENGTH,
            )

            with torch.no_grad():
                outputs = model(
                    **model_inputs,
                    use_cache=False,
                    return_dict=True,
                )
                emotion_logits = outputs.logits[
                    :, -1, list(emotion_token_ids)
                ].float()

            prediction_indices = emotion_logits.cpu().argmax(dim=-1).numpy()
            all_predictions.extend(emotions[index] for index in prediction_indices)

            del model_inputs, outputs, emotion_logits
            cleanup_memory()
    finally:
        if hook_handle is not None:
            hook_handle.remove()

    df_out[pred_col] = all_predictions
    df_out[f"{pred_col}_correct"] = (
        df_out[label_col].astype(str) == df_out[pred_col]
    )
    metrics = compute_classification_metrics(
        true_labels=df_out[label_col].astype(str).tolist(),
        predicted_labels=df_out[pred_col].astype(str).tolist(),
        emotions=emotions,
    )
    return df_out, metrics


# ============================================================
# General-language perplexity matched to feature_steering_utils.py
# ============================================================

def load_general_texts_wikitext(
    dataset_name: str = "wikitext",
    dataset_config: str = "wikitext-103-raw-v1",
    split: str = "test",
    cache_dir: Optional[str] = None,
    min_chars: int = 100,
    max_texts: int = 500,
) -> List[str]:
    """
    Match feature_steering_utils.load_general_texts_wikitext().
    """
    from datasets import load_dataset

    wiki = load_dataset(
        dataset_name,
        dataset_config,
        split=split,
        cache_dir=cache_dir,
    )

    texts = [
        text
        for text in wiki["text"]
        if len(str(text).strip()) > min_chars
    ]

    return texts[:max_texts]


def get_underlying_transformer_layers(llm):
    """
    Locate the real Hugging Face transformer layer modules.
    """
    model = getattr(llm, "_model", None)

    if model is None:
        raise RuntimeError(
            "Could not access the underlying Hugging Face model."
        )

    candidate_paths = [
        ("model", "layers"),
        ("model", "language_model", "layers"),
        ("language_model", "model", "layers"),
        ("language_model", "layers"),
        ("transformer", "h"),
        ("gpt_neox", "layers"),
    ]

    for path in candidate_paths:
        current = model

        try:
            for attribute in path:
                current = getattr(current, attribute)
        except AttributeError:
            continue

        try:
            _ = current[0]
            return current
        except Exception:
            continue

    raise RuntimeError(
        "Could not locate the underlying transformer layers."
    )


def make_all_token_projection_scaling_hook(
    directions: torch.Tensor,
    center: torch.Tensor,
    scales: torch.Tensor,
):
    """
    Apply learned vector projection scaling to all sequence positions.

    This matches the SAE perplexity test's all-token intervention mode.
    """
    detached_directions = directions.detach()
    detached_center = center.detach()
    detached_scales = scales.detach()

    def hook(module, inputs, output):
        if isinstance(output, tuple):
            hidden = output[0]
        else:
            hidden = output

        hidden_float = hidden.float()
        centered_hidden = (
            hidden_float
            - detached_center.view(1, 1, -1)
        )

        projections = centered_hidden @ detached_directions.T

        scaled_change = projections * (
            detached_scales - 1.0
        ).view(1, 1, -1)

        delta = scaled_change @ detached_directions
        modified_hidden = (
            hidden_float + delta
        ).to(hidden.dtype)

        if isinstance(output, tuple):
            return (modified_hidden,) + output[1:]

        return modified_hidden

    return hook


def tokenize_for_perplexity(
    llm,
    texts: List[str],
    max_length: int = 512,
):
    """
    Match SAEFeatureFinetuner.tokenize().
    """
    return llm.tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )


@torch.no_grad()
def evaluate_general_perplexity(
    llm,
    texts: List[str],
    directions: torch.Tensor,
    center: torch.Tensor,
    scales: torch.Tensor,
    layer_id: int,
    batch_size: int = 8,
    max_length: int = 512,
    apply_steering: bool = False,
) -> Dict[str, float]:
    """
    Match SAEFeatureFinetuner.evaluate_perplexity_clean() and
    SAEFeatureFinetuner.evaluate_perplexity().

    The loss weighting intentionally counts all non-padding labels,
    exactly as in feature_steering_utils.py, so the resulting numbers
    are directly comparable to that pipeline.
    """
    model = getattr(llm, "_model", None)

    if model is None:
        raise RuntimeError(
            "Could not access the underlying Hugging Face model."
        )

    model.eval()
    device = get_model_device(llm)
    hook_handle = None

    if apply_steering:
        actual_layers = get_underlying_transformer_layers(llm)

        if layer_id < 0 or layer_id >= len(actual_layers):
            raise ValueError(
                f"layer_id={layer_id} is invalid for the model."
            )

        hook_handle = actual_layers[layer_id].register_forward_hook(
            make_all_token_projection_scaling_hook(
                directions=directions,
                center=center,
                scales=scales,
            )
        )

    total_loss = 0.0
    total_tokens = 0

    try:
        for start in tqdm(
            range(0, len(texts), batch_size),
            desc=(
                "ppl steered"
                if apply_steering
                else "ppl clean"
            ),
            leave=False,
        ):
            batch_texts = texts[start:start + batch_size]

            inputs = tokenize_for_perplexity(
                llm=llm,
                texts=batch_texts,
                max_length=max_length,
            )

            input_ids = inputs["input_ids"].to(device)
            attention_mask = inputs["attention_mask"].to(device)

            labels = input_ids.clone()
            labels[attention_mask == 0] = -100

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            # Keep this identical to feature_steering_utils.py.
            num_tokens = int((labels != -100).sum().item())
            total_loss += float(outputs.loss.item()) * num_tokens
            total_tokens += num_tokens

            del input_ids, attention_mask, labels, outputs
            cleanup_memory()

    finally:
        if hook_handle is not None:
            hook_handle.remove()

    avg_loss = total_loss / max(total_tokens, 1)

    return {
        "perplexity": float(math.exp(avg_loss)),
        "avg_loss": float(avg_loss),
        "total_tokens": int(total_tokens),
        "num_texts": int(len(texts)),
        "max_length": int(max_length),
        "all_token_intervention": bool(apply_steering),
    }


# ============================================================
# Train six scalers
# ============================================================

def train_emotion_scalers(
    llm,
    layers,
    train_df: pd.DataFrame,
    directions: torch.Tensor,
    center: torch.Tensor,
    layer_id: int,
    emotions: Sequence[str],
    emotion_token_ids: Sequence[int],
    text_col: str,
    label_col: str,
    batch_size: int,
    num_steps: int,
    learning_rate: float,
    scale_regularization: float,
    min_scale: float,
    max_scale: float,
    seed: int,
    log_every: int,
) -> Tuple[torch.Tensor, pd.DataFrame]:
    """
    Train the same six positive scalar parameters and objective as the
    source notebook, using an autograd-preserving PyTorch forward hook.
    """
    freeze_llm_parameters(llm)
    model = getattr(llm, "_model", None)
    if model is None:
        raise RuntimeError("Could not access the underlying Hugging Face model.")

    model.eval()
    device = directions.device
    label_to_index = {emotion: i for i, emotion in enumerate(emotions)}

    # exp(0) = 1, so step 0 is exactly the unsteered model.
    log_scales = nn.Parameter(
        torch.zeros(len(emotions), dtype=torch.float32, device=device)
    )
    optimizer = torch.optim.AdamW(
        [log_scales], lr=learning_rate, weight_decay=0.0
    )

    def steering_hook(_module, _inputs, layer_output):
        residual = get_hidden_tensor_from_layer_output(layer_output)
        final_hidden = residual[:, -1, :]
        steered_final = apply_projection_scaling(
            final_hidden_float=final_hidden.float(),
            directions=directions,
            center=center,
            scales=torch.exp(log_scales),
        ).to(final_hidden.dtype)
        modified = torch.cat(
            [residual[:, :-1, :], steered_final.unsqueeze(1)],
            dim=1,
        )
        return replace_hidden_tensor_in_layer_output(layer_output, modified)

    hook_handle = layers[layer_id].register_forward_hook(steering_hook)
    rng = np.random.default_rng(seed)
    history_rows: List[Dict[str, float]] = []
    shuffled_indices = rng.permutation(len(train_df))
    cursor = 0

    try:
        for step in tqdm(
            range(1, num_steps + 1),
            desc="Training six emotion scalers",
        ):
            if cursor + batch_size > len(shuffled_indices):
                shuffled_indices = rng.permutation(len(train_df))
                cursor = 0

            batch_indices = shuffled_indices[cursor:cursor + batch_size]
            cursor += batch_size
            if len(batch_indices) == 0:
                raise RuntimeError("An empty training batch was created.")

            batch_df = train_df.iloc[batch_indices]
            batch_inputs = batch_df[text_col].astype(str).tolist()
            model_inputs = tokenize_model_batch(llm, batch_inputs, device)
            target_labels = torch.tensor(
                [label_to_index[label] for label in batch_df[label_col].astype(str)],
                dtype=torch.long,
                device=device,
            )

            optimizer.zero_grad(set_to_none=True)
            outputs = model(
                **model_inputs,
                use_cache=False,
                return_dict=True,
            )
            emotion_logits = outputs.logits[
                :, -1, list(emotion_token_ids)
            ].float()
            scales = torch.exp(log_scales)
            classification_loss = F.cross_entropy(
                emotion_logits, target_labels
            )
            regularization_loss = ((scales - 1.0) ** 2).mean()
            loss = (
                classification_loss
                + scale_regularization * regularization_loss
            )

            loss_value = float(loss.detach().cpu().item())
            classification_loss_value = float(
                classification_loss.detach().cpu().item()
            )
            regularization_loss_value = float(
                regularization_loss.detach().cpu().item()
            )
            loss.backward()

            if log_scales.grad is None:
                raise RuntimeError(
                    "No gradient reached the six scaler parameters."
                )

            torch.nn.utils.clip_grad_norm_([log_scales], max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                log_scales.clamp_(
                    min=math.log(min_scale),
                    max=math.log(max_scale),
                )

            current_scales = torch.exp(log_scales.detach())
            history_row: Dict[str, float] = {
                "step": float(step),
                "loss": loss_value,
                "classification_loss": classification_loss_value,
                "regularization_loss": regularization_loss_value,
            }
            for emotion_index, emotion in enumerate(emotions):
                history_row[f"{emotion}_scale"] = float(
                    current_scales[emotion_index].cpu().item()
                )
            history_rows.append(history_row)

            if step == 1 or step == num_steps or step % log_every == 0:
                scale_string = ", ".join(
                    f"{emotion}={current_scales[index].item():.3f}"
                    for index, emotion in enumerate(emotions)
                )
                print(
                    f"Step {step:>4}/{num_steps}: "
                    f"loss={history_row['loss']:.4f}; {scale_string}"
                )

            del model_inputs, outputs, emotion_logits, loss
    finally:
        hook_handle.remove()

    learned_scales = torch.exp(log_scales.detach())
    history_df = pd.DataFrame(history_rows)
    return learned_scales, history_df


# ============================================================
# Full pipeline
# ============================================================


# ============================================================
# External-dataset normalization and reporting
# ============================================================

def load_external_dataframe(
    dataset_name: str,
    spec: Dict[str, str],
    canonical_text_col: str,
    canonical_label_col: str,
    emotions: Sequence[str],
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """Load one external dataset without using it for training."""
    csv_path = spec["path"]

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"External CSV not found: {csv_path}")

    raw_df = pd.read_csv(csv_path, sep=spec.get("sep", ","))
    required = [spec["text_col"], spec["label_col"]]
    missing = [column for column in required if column not in raw_df.columns]

    if missing:
        raise ValueError(
            f"{dataset_name} ({csv_path}) is missing columns {missing}. "
            f"Available columns: {raw_df.columns.tolist()}"
        )

    normalized = pd.DataFrame({
        canonical_text_col: raw_df[spec["text_col"]],
        canonical_label_col: raw_df[spec["label_col"]],
        "source_row": np.arange(len(raw_df)),
    })

    missing_mask = (
        normalized[canonical_text_col].isna()
        | normalized[canonical_label_col].isna()
    )
    normalized = normalized.loc[~missing_mask].copy()
    normalized[canonical_text_col] = (
        normalized[canonical_text_col].astype(str)
    )
    normalized[canonical_label_col] = (
        normalized[canonical_label_col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    empty_text_mask = (
        normalized[canonical_text_col].str.strip().str.len() == 0
    )
    normalized = normalized.loc[~empty_text_mask].copy()

    label_counts_before_filter = (
        normalized[canonical_label_col].value_counts().sort_index().to_dict()
    )
    supported_mask = normalized[canonical_label_col].isin(emotions)
    filtered_label_counts = (
        normalized.loc[~supported_mask, canonical_label_col]
        .value_counts()
        .sort_index()
        .to_dict()
    )
    normalized = normalized.loc[supported_mask].reset_index(drop=True)

    apply_prompt_template = bool(
        spec.get("apply_prompt_template", False)
    )
    if apply_prompt_template:
        normalized[canonical_text_col] = normalized[
            canonical_text_col
        ].map(
            lambda text: (
                f"In this text: {text}, the emotion implied is:"
            )
        )

    missing_emotions = [
        emotion
        for emotion in emotions
        if emotion not in set(normalized[canonical_label_col])
    ]
    if missing_emotions:
        print(
            f"Warning: {dataset_name} has no retained examples for "
            f"{missing_emotions}; their F1 values will be zero."
        )

    summary = {
        "dataset": dataset_name,
        "path": csv_path,
        "raw_rows": int(len(raw_df)),
        "missing_value_rows_removed": int(missing_mask.sum()),
        "empty_text_rows_removed": int(empty_text_mask.sum()),
        "unsupported_label_rows_removed": int((~supported_mask).sum()),
        "evaluated_rows": int(len(normalized)),
        "prompt_template_applied": apply_prompt_template,
        "classification_max_length": CLASSIFICATION_MAX_LENGTH,
        "labels_before_filter": json.dumps(label_counts_before_filter),
        "filtered_labels": json.dumps(filtered_label_counts),
    }

    if normalized.empty:
        raise ValueError(
            f"No supported evaluation rows remain in {dataset_name}."
        )

    print(f"\n{dataset_name} preprocessing summary:")
    print(json.dumps(summary, indent=2))
    print("Retained label counts:")
    print(
        normalized[canonical_label_col]
        .value_counts()
        .reindex(emotions, fill_value=0)
    )

    return normalized, summary


def metrics_to_long_f1(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """Convert wide macro/per-emotion F1 columns into tidy rows."""
    id_columns = ["dataset", "method", "model_name", "layer_id"]
    f1_columns = ["macro_f1"] + [f"{emotion}_f1" for emotion in EMOTIONS]
    long_df = metrics_df.melt(
        id_vars=id_columns,
        value_vars=f1_columns,
        var_name="metric",
        value_name="f1",
    )
    long_df["emotion"] = long_df["metric"].map(
        lambda value: "macro" if value == "macro_f1" else value[:-3]
    )
    return long_df[id_columns + ["emotion", "f1"]]


def run_pipeline(args: argparse.Namespace) -> Dict[str, object]:
    set_seed(args.seed)

    train_df = load_and_validate_dataframe(
        csv_path=args.train_csv,
        text_col=args.text_col,
        label_col=args.label_col,
        emotions=EMOTIONS,
        require_all_emotions=True,
    )

    external_dfs = {}
    dataset_summaries = []
    for dataset_name, spec in args.external_datasets.items():
        external_df, summary = load_external_dataframe(
            dataset_name=dataset_name,
            spec=spec,
            canonical_text_col=args.text_col,
            canonical_label_col=args.label_col,
            emotions=EMOTIONS,
        )
        external_dfs[dataset_name] = external_df
        dataset_summaries.append(summary)

    print(f"\nMain training examples: {len(train_df)}")
    print("Training label counts:")
    print(train_df[args.label_col].value_counts().reindex(EMOTIONS))

    device_name = get_device_name()
    dtype = choose_dtype(device_name)

    print(f"Loading model: {args.model_name}")
    print(f"Device: {device_name}")
    print(f"Dtype: {dtype}")

    llm = load_llm(
        model_name=args.model_name,
        device_map=device_name,
        dtype=dtype,
    )

    layers, n_layers, hidden_dim = get_model_layers_and_dims(llm)
    validate_layer_id(args.layer_id, n_layers)

    print(
        f"Model has {n_layers} layers and hidden size {hidden_dim}. "
        f"Using layer {args.layer_id}."
    )

    _, emotion_to_token = build_emotion_token_maps(
        llm=llm,
        emotions=EMOTIONS,
        verbose=True,
    )
    emotion_token_ids = [emotion_to_token[e] for e in EMOTIONS]

    safe_name = model_safe_name(args.model_name)
    vector_dir = (
        Path(args.output_dir)
        / safe_name
        / "emotion_vector_steering"
        / f"layer_{args.layer_id}"
    )
    eval_dir = vector_dir / "external_generalizability"
    vector_dir.mkdir(parents=True, exist_ok=True)
    eval_dir.mkdir(parents=True, exist_ok=True)

    # These cache paths and metadata deliberately match the supplied notebook,
    # preserving its hidden-state extraction and concept-vector process.
    hidden_path = vector_dir / "train_hidden_states.npy"
    hidden_metadata_path = vector_dir / "train_hidden_metadata.json"
    cache_metadata = {
        "model_name": args.model_name,
        "layer_id": args.layer_id,
        "hidden_dim": hidden_dim,
        "num_examples": len(train_df),
        "text_col": args.text_col,
        "label_col": args.label_col,
        "data_fingerprint": dataframe_fingerprint(
            train_df, [args.text_col, args.label_col]
        ),
    }

    train_hidden_states = load_or_extract_train_hidden_states(
        llm=llm,
        layers=layers,
        train_df=train_df,
        layer_id=args.layer_id,
        text_col=args.text_col,
        batch_size=args.hidden_batch_size,
        hidden_dim=hidden_dim,
        hidden_path=hidden_path,
        metadata_path=hidden_metadata_path,
        cache_metadata=cache_metadata,
        overwrite_hidden=args.overwrite_hidden,
    )

    train_labels = train_df[args.label_col].to_numpy()
    direction_array, center_array = compute_contrastive_emotion_vectors(
        train_hidden_states=train_hidden_states,
        train_labels=train_labels,
        emotions=EMOTIONS,
    )

    np.save(vector_dir / "emotion_vectors.npy", direction_array)
    np.save(vector_dir / "train_hidden_center.npy", center_array)
    save_direction_diagnostics(
        directions=direction_array,
        emotions=EMOTIONS,
        output_dir=vector_dir,
    )

    device = get_model_device(llm)
    directions = F.normalize(
        torch.tensor(direction_array, dtype=torch.float32, device=device),
        dim=-1,
    )
    center = torch.tensor(center_array, dtype=torch.float32, device=device)
    baseline_scales = torch.ones(
        len(EMOTIONS), dtype=torch.float32, device=device
    )

    # Train scalers on the main dataset only, exactly as in the supplied
    # notebook. External labels are never exposed during training.
    learned_scales, history_df = train_emotion_scalers(
        llm=llm,
        layers=layers,
        train_df=train_df,
        directions=directions,
        center=center,
        layer_id=args.layer_id,
        emotions=EMOTIONS,
        emotion_token_ids=emotion_token_ids,
        text_col=args.text_col,
        label_col=args.label_col,
        batch_size=min(args.train_batch_size, len(train_df)),
        num_steps=args.num_steps,
        learning_rate=args.learning_rate,
        scale_regularization=args.scale_regularization,
        min_scale=args.min_scale,
        max_scale=args.max_scale,
        seed=args.seed,
        log_every=args.log_every,
    )

    metric_rows = []
    prediction_frames = {}

    for dataset_name, external_df in external_dfs.items():
        baseline_df, baseline_metrics = evaluate_emotion_vector_steering(
            llm=llm,
            layers=layers,
            df=external_df,
            directions=directions,
            center=center,
            scales=baseline_scales,
            layer_id=args.layer_id,
            emotions=EMOTIONS,
            emotion_token_ids=emotion_token_ids,
            text_col=args.text_col,
            label_col=args.label_col,
            batch_size=args.eval_batch_size,
            pred_col="baseline_pred_emotion",
        )
        steered_df, steered_metrics = evaluate_emotion_vector_steering(
            llm=llm,
            layers=layers,
            df=external_df,
            directions=directions,
            center=center,
            scales=learned_scales,
            layer_id=args.layer_id,
            emotions=EMOTIONS,
            emotion_token_ids=emotion_token_ids,
            text_col=args.text_col,
            label_col=args.label_col,
            batch_size=args.eval_batch_size,
            pred_col="steered_pred_emotion",
        )

        for method, metrics in [
            ("unsteered", baseline_metrics),
            ("emotion_vector_steering", steered_metrics),
        ]:
            metric_rows.append({
                "dataset": dataset_name,
                "method": method,
                "model_name": args.model_name,
                "layer_id": args.layer_id,
                "n_examples": len(external_df),
                **metrics,
            })

        predictions = external_df.copy()
        predictions["dataset"] = dataset_name
        predictions["baseline_pred_emotion"] = baseline_df[
            "baseline_pred_emotion"
        ]
        predictions["baseline_correct"] = baseline_df[
            "baseline_pred_emotion_correct"
        ]
        predictions["steered_pred_emotion"] = steered_df[
            "steered_pred_emotion"
        ]
        predictions["steered_correct"] = steered_df[
            "steered_pred_emotion_correct"
        ]
        prediction_frames[dataset_name] = predictions

        print(f"\n{dataset_name} unsteered metrics:")
        print(json.dumps(baseline_metrics, indent=2))
        print(f"\n{dataset_name} steered metrics:")
        print(json.dumps(steered_metrics, indent=2))

    metrics_df = pd.DataFrame(metric_rows)
    f1_long_df = metrics_to_long_f1(metrics_df)
    scales_df = pd.DataFrame({
        "emotion": EMOTIONS,
        "scale": learned_scales.detach().cpu().numpy(),
        "layer_id": args.layer_id,
        "model_name": args.model_name,
    })
    summary_df = pd.DataFrame(dataset_summaries)

    perplexity_df = pd.DataFrame()
    if args.run_perplexity_check:
        if not args.use_wikitext:
            raise ValueError(
                "Comparable perplexity evaluation expects USE_WIKITEXT=True."
            )
        general_texts = load_general_texts_wikitext(
            dataset_name=args.wikitext_dataset_name,
            dataset_config=args.wikitext_dataset_config,
            split=args.wikitext_split,
            cache_dir=args.wikitext_cache_dir,
            min_chars=args.ppl_min_chars,
            max_texts=args.ppl_max_texts,
        )
        baseline_ppl = evaluate_general_perplexity(
            llm=llm,
            texts=general_texts,
            directions=directions,
            center=center,
            scales=baseline_scales,
            layer_id=args.layer_id,
            batch_size=args.ppl_batch_size,
            max_length=args.ppl_max_length,
            apply_steering=False,
        )
        steered_ppl = evaluate_general_perplexity(
            llm=llm,
            texts=general_texts,
            directions=directions,
            center=center,
            scales=learned_scales,
            layer_id=args.layer_id,
            batch_size=args.ppl_batch_size,
            max_length=args.ppl_max_length,
            apply_steering=True,
        )
        perplexity_df = pd.DataFrame([
            {
                "method": "unsteered",
                "model_name": args.model_name,
                "layer_id": args.layer_id,
                **baseline_ppl,
            },
            {
                "method": "emotion_vector_steering",
                "model_name": args.model_name,
                "layer_id": args.layer_id,
                **steered_ppl,
            },
        ])

    scales_df.to_csv(eval_dir / "learned_emotion_scales.csv", index=False)
    history_df.to_csv(eval_dir / "training_history.csv", index=False)
    metrics_df.to_csv(eval_dir / "external_metrics.csv", index=False)
    f1_long_df.to_csv(eval_dir / "external_f1_long.csv", index=False)
    summary_df.to_csv(eval_dir / "dataset_preprocessing_summary.csv", index=False)
    for dataset_name, predictions in prediction_frames.items():
        predictions.to_csv(
            eval_dir / f"{dataset_name}_predictions.csv", index=False
        )
    if not perplexity_df.empty:
        perplexity_df.to_csv(
            eval_dir / "perplexity_diagnostic.csv", index=False
        )

    checkpoint = {
        "model_name": args.model_name,
        "layer_id": args.layer_id,
        "emotions": EMOTIONS,
        "learned_scales": learned_scales.detach().cpu(),
        "emotion_vectors": directions.detach().cpu(),
        "train_hidden_center": center.detach().cpu(),
        "text_col": args.text_col,
        "label_col": args.label_col,
        "evaluation_datasets": list(args.external_datasets),
        "intervention": (
            "h' = h + sum_e (scale_e - 1) * "
            "<h - train_mean, v_e> * v_e"
        ),
    }
    torch.save(
        checkpoint,
        eval_dir / "emotion_vector_steering_checkpoint.pt",
    )

    config_dict = vars(args).copy()
    config_dict["emotions"] = EMOTIONS
    config_dict["n_layers"] = n_layers
    config_dict["hidden_dim"] = hidden_dim
    with open(eval_dir / "run_config.json", "w") as file:
        json.dump(config_dict, file, indent=2)

    print(f"\nSaved external generalizability outputs to: {eval_dir}")
    print("\nExternal-dataset F1 results:")
    print(f1_long_df.to_string(index=False))

    output = {
        "metrics": metrics_df,
        "f1_long": f1_long_df,
        "scales": scales_df,
        "history": history_df,
        "predictions": prediction_frames,
        "dataset_summary": summary_df,
        "perplexity": perplexity_df,
        "output_dir": str(eval_dir),
    }

    del llm, layers, directions, center, baseline_scales, learned_scales
    cleanup_memory()
    return output


In [ ]:
args = argparse.Namespace(
    model_name=MODEL_NAME,
    layer_id=LAYER_ID,
    train_csv=TRAIN_CSV,
    external_datasets=EXTERNAL_DATASETS,
    output_dir=OUTPUT_DIR,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    hf_cache=HF_CACHE,
    hidden_batch_size=HIDDEN_BATCH_SIZE,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    num_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE,
    scale_regularization=SCALE_REGULARIZATION,
    min_scale=MIN_SCALE,
    max_scale=MAX_SCALE,
    seed=SEED,
    log_every=LOG_EVERY,
    overwrite_hidden=OVERWRITE_HIDDEN,
    run_perplexity_check=RUN_PERPLEXITY_CHECK,
    use_wikitext=USE_WIKITEXT,
    wikitext_dataset_name=WIKITEXT_DATASET_NAME,
    wikitext_dataset_config=WIKITEXT_DATASET_CONFIG,
    wikitext_split=WIKITEXT_SPLIT,
    wikitext_cache_dir=WIKITEXT_CACHE_DIR,
    ppl_min_chars=PPL_MIN_CHARS,
    ppl_max_texts=PPL_MAX_TEXTS,
    ppl_batch_size=PPL_BATCH_SIZE,
    ppl_max_length=PPL_MAX_LENGTH,
)

results = run_pipeline(args)

display(results["dataset_summary"])
display(results["metrics"])
display(results["f1_long"])
display(results["scales"])

if not results["perplexity"].empty:
    display(results["perplexity"])


In [ ]:
# Optional: run this cell if you interrupt the pipeline or want to clear GPU cache.
cleanup_memory()
print("Python and CUDA caches cleared.")


In [ ]:
# ============================================================
# Aggregate and print external-evaluation results across models
# Run this cell after completing the notebook for each model.
# ============================================================

metric_paths = sorted(
    Path(OUTPUT_DIR).glob(
        "*/emotion_vector_steering/layer_*/"
        "external_generalizability/external_metrics.csv"
    )
)

if not metric_paths:
    print(f"No completed external evaluations found under {OUTPUT_DIR!r}.")
else:
    all_model_metrics = pd.concat(
        [pd.read_csv(path) for path in metric_paths],
        ignore_index=True,
    ).sort_values(
        ["model_name", "layer_id", "dataset", "method"]
    ).reset_index(drop=True)

    metric_columns = [
        "model_name", "layer_id", "dataset", "method",
        "n_examples", "accuracy", "macro_f1",
        *[f"{emotion}_f1" for emotion in EMOTIONS],
    ]
    all_model_metrics = all_model_metrics[metric_columns]

    print(f"Loaded {len(metric_paths)} completed model runs.")
    print("\nAll external evaluation metrics:")
    print(all_model_metrics.to_string(index=False))

    macro_f1_comparison = all_model_metrics.pivot_table(
        index=["model_name", "layer_id"],
        columns=["dataset", "method"],
        values="macro_f1",
    )
    print("\nMacro F1 comparison:")
    display(macro_f1_comparison)

    all_model_f1_long = metrics_to_long_f1(all_model_metrics)
    display(all_model_f1_long)

    combined_metrics_path = (
        Path(OUTPUT_DIR) / "all_models_external_metrics.csv"
    )
    combined_f1_path = (
        Path(OUTPUT_DIR) / "all_models_external_f1_long.csv"
    )
    all_model_metrics.to_csv(combined_metrics_path, index=False)
    all_model_f1_long.to_csv(combined_f1_path, index=False)
    print(f"\nSaved combined metrics to: {combined_metrics_path}")
    print(f"Saved combined F1 table to: {combined_f1_path}")
